In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# SQL Server to Databricks Pipeline Generation

This notebook generates optimized SQL Server to Databricks ingestion pipelines using Databricks Asset Bundles.

## Process Overview
1. **Load balancing**: Groups tables into pipeline configurations
2. **YAML generation**: Creates Databricks Asset Bundle YAML files

## Prerequisites
- Upload the `load_balancing` and `deployment` directories to your Databricks workspace
- Ensure you have a CSV file with your source table list
- Have a Databricks connection configured for SQL Server

## Install Required Libraries

Uncomment and run if needed:

In [0]:
# %pip install -r requirements.txt 

## Import Required Modules

In [0]:
import pandas as pd
import os
from databricks.sdk import WorkspaceClient
from load_balancing.generate_pipeline_config import generate_pipeline_config
from deployment.generate_dab_yaml import generate_yaml_files

## Define Pipeline Generation Function

In [ ]:
def run_complete_pipeline_generation(
    input_csv: str,
    gateway_catalog: str,
    gateway_schema: str,
    project_name: str,
    output_dir: str,
    workspace_host: str,
    root_path: str,
    default_connection_name: str,
    max_tables_per_group: int = 250,
    default_schedule: str = "*/15 * * * *",
    node_type_id: str = None,
    driver_node_type_id: str = None
):
    """
    Complete pipeline generation process from source table list to YAML files.

    Args:
        input_csv (str): Path to input CSV with source table list
        gateway_catalog (str): Catalog for gateway storage metadata
        gateway_schema (str): Schema for gateway storage metadata
        project_name (str): Project name prefix for all resources
        output_dir (str): Output directory for DAB project
        workspace_host (str): Workspace host URL
        root_path (str): Root path for bundle deployment
        default_connection_name (str): Default connection name
        max_tables_per_group (int): Maximum tables per pipeline group (default: 250)
        default_schedule (str): Default cron schedule (default: "*/15 * * * *")
        node_type_id (str): Worker node type (optional)
        driver_node_type_id (str): Driver node type (optional)

    Returns:
        pd.DataFrame: The pipeline configuration dataframe
    """
    print("="*80)
    print("STARTING COMPLETE PIPELINE GENERATION PROCESS")
    print("="*80)

    # Step 1: Load input CSV
    print(f"\n[Step 1/4] Loading input CSV: {input_csv}")
    input_df = pd.read_csv(input_csv)
    print(f"  ✓ Loaded {len(input_df)} tables from {input_df['source_database'].nunique()} databases")

    # Step 2: Generate pipeline configuration (load balancing)
    print(f"\n[Step 2/4] Generating pipeline configuration with load balancing")
    print(f"  - Max tables per group: {max_tables_per_group}")
    print(f"  - Connection name: {default_connection_name}")
    print(f"  - Schedule: {default_schedule}")

    pipeline_config_df = generate_pipeline_config(
        df=input_df,
        max_tables_per_group=max_tables_per_group,
        default_connection_name=default_connection_name,
        default_schedule=default_schedule
    )

    # Step 3: Generate YAML files
    print(f"\n[Step 3/4] Generating Databricks Asset Bundle YAML files")
    print(f"  - Gateway catalog: {gateway_catalog}")
    print(f"  - Gateway schema: {gateway_schema}")
    print(f"  - Project name: {project_name}")
    print(f"  - Output directory: {output_dir}")
    print(f"  - Root path: {root_path}")

    generate_yaml_files(
        df=pipeline_config_df,
        gateway_catalog=gateway_catalog,
        gateway_schema=gateway_schema,
        project_name=project_name,
        node_type_id=node_type_id,
        driver_node_type_id=driver_node_type_id,
        output_dir=output_dir,
        workspace_host=workspace_host,
        root_path=root_path
    )

    print("\n" + "="*80)
    print("PIPELINE GENERATION COMPLETE!")
    print("="*80)
    print(f"\nNext steps:")
    print(f"  1. Review the generated DAB project:")
    print(f"     - {output_dir}/databricks.yml")
    print(f"     - {output_dir}/resources/gateways.yml")
    print(f"     - {output_dir}/resources/pipelines.yml")
    print(f"  2. Deploy using Databricks Asset Bundles:")
    print(f"     cd {output_dir}")
    print(f"     databricks bundle deploy -t <environment>")
    print("="*80)

    return pipeline_config_df

## Configure Parameters

**IMPORTANT**: Modify these parameters for your environment before running!

In [0]:
# Input CSV configuration
INPUT_CSV = 'load_balancing/examples/tapworks_config.csv'  # Update with your CSV path

# Databricks configuration
GATEWAY_CATALOG = 'tapworks_catalog'        # Update with your catalog name
GATEWAY_SCHEMA = 'tapworks'   # Update with your schema name
PROJECT_NAME = 'tapworks_project'           # Update with your project name

# Pipeline configuration
MAX_TABLES_PER_GROUP = 1000           # Maximum tables per pipeline group
DEFAULT_CONNECTION_NAME = 'tapworks_sql_connection'    # Your Databricks connection name
DEFAULT_SCHEDULE = '*/15 * * * *'     # Cron schedule (every 15 minutes)

# Cluster configuration
# keep this for reference
# NODE_TYPE_ID = 'm5d.large'            # Worker node type
# DRIVER_NODE_TYPE_ID = 'c5a.8xlarge'   # Driver node type


# Initialize workspace client
w = WorkspaceClient()

# Automatically get current user email
username = w.current_user.me().user_name

# Automatically get workspace host URL
workspace_host = w.config.host

# Automatically construct output directory
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/test_sql'
WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'


## Alternative: Use Databricks Widgets for Interactive Configuration

Uncomment and run this cell to use widgets instead of hardcoded values:

In [0]:
# # Create widgets for interactive configuration
# dbutils.widgets.text("input_csv", "load_balancing/examples/example_config.csv", "Input CSV Path")
# dbutils.widgets.text("gateway_catalog", "jack_demos", "Gateway Catalog")
# dbutils.widgets.text("gateway_schema", "ingestion_schema", "Gateway Schema")
# dbutils.widgets.text("project_name", "my_project", "Project Name")
# dbutils.widgets.text("connection_name", "conn_1", "Connection Name")
# dbutils.widgets.text("max_tables", "1000", "Max Tables Per Group")
# dbutils.widgets.text("output_dir", "/Workspace/Users/your-email/dab_project", "Output Directory")

# # Get widget values
# INPUT_CSV = dbutils.widgets.get("input_csv")
# GATEWAY_CATALOG = dbutils.widgets.get("gateway_catalog")
# GATEWAY_SCHEMA = dbutils.widgets.get("gateway_schema")
# PROJECT_NAME = dbutils.widgets.get("project_name")
# DEFAULT_CONNECTION_NAME = dbutils.widgets.get("connection_name")
# MAX_TABLES_PER_GROUP = int(dbutils.widgets.get("max_tables"))
# OUTPUT_DIR = dbutils.widgets.get("output_dir")

## Run Pipeline Generation

Execute the complete pipeline generation process:

In [ ]:
result_df = run_complete_pipeline_generation(
    input_csv=INPUT_CSV,
    gateway_catalog=GATEWAY_CATALOG,
    gateway_schema=GATEWAY_SCHEMA,
    project_name=PROJECT_NAME,
    max_tables_per_group=MAX_TABLES_PER_GROUP,
    default_connection_name=DEFAULT_CONNECTION_NAME,
    default_schedule=DEFAULT_SCHEDULE,
    # node_type_id=NODE_TYPE_ID,
    # driver_node_type_id=DRIVER_NODE_TYPE_ID,
    output_dir=OUTPUT_DIR,
    workspace_host=WORKSPACE_HOST,
    root_path=ROOT_PATH
)

## Review Generated Configuration

Display the generated pipeline configuration:

In [0]:
display(result_df)

## Summary Statistics

In [0]:
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total tables: {len(result_df)}")
print(f"Total databases: {result_df['source_database'].nunique()}")
print(f"Total gateways: {result_df['gateway'].nunique()}")
print(f"Total pipeline groups: {result_df['pipeline_group'].nunique()}")
print(f"\nTables per pipeline group:")
print(result_df.groupby('pipeline_group').size())
print("="*80)

## Optional: Save Intermediate Configuration

Save the pipeline configuration to a CSV file for reference:

In [0]:
# Uncomment to save the intermediate configuration
# output_csv = f"{OUTPUT_DIR}/generated_config.csv"
# result_df.to_csv(output_csv, index=False)
# print(f"✓ Intermediate configuration saved to: {output_csv}")

## View Generated Files

Check the generated DAB project structure:

In [0]:
# List generated files
print("\nGenerated DAB Project Structure:")
print(f"\n{OUTPUT_DIR}/")

# Check if files exist
import os
files_to_check = [
    'databricks.yml',
    'resources/gateways.yml',
    'resources/pipelines.yml'
]

for file in files_to_check:
    full_path = os.path.join(OUTPUT_DIR, file)
    exists = "✓" if os.path.exists(full_path) else "✗"
    print(f"  {exists} {file}")

## Next Steps

1. **Review Generated Files**: Check the YAML files in the output directory
2. **Download DAB Project**: Download the generated DAB project to your local machine
3. **Deploy**:
   ```bash
   cd <output_dir>
   databricks bundle validate
   databricks bundle deploy -t dev
   ```
4. **Monitor**: Check your Databricks workspace for the deployed pipelines and gateways